In [6]:
use std::fs::File;
use std::io::{Read, Seek, SeekFrom};
use std::time::Instant;

// ===== CRC32テーブル (コンパイル時定数) =====
const CRC_TABLE: [u32; 256] = {
    let mut table = [0u32; 256];
    let mut i = 0;
    while i < 256 {
        let mut c = i as u32;
        let mut j = 0;
        while j < 8 {
            c = if c & 1 != 0 { (c >> 1) ^ 0xedb88320 } else { c >> 1 };
            j += 1;
        }
        table[i] = c;
        i += 1;
    }
    table
};

fn crc32(c: u32, d: u8) -> u32 {
    (c >> 8) ^ CRC_TABLE[((c ^ d as u32) & 0xff) as usize]
}

fn decrypt_byte(key2: u32) -> u8 {
    let temp = (key2 | 2) as u16;
    ((u32::from(temp) * u32::from(temp ^ 1)) >> 8) as u8
}

fn msb(x: u32) -> u8 { (x >> 24) as u8 }

// ===== Key構造体 (attack6で使用) =====
#[derive(Clone, Copy)]
struct Key { key0: u32, key1: u32, key2: u32 }

impl Key {
    fn new(k0: u32, k1: u32, k2: u32) -> Self { Key { key0: k0, key1: k1, key2: k2 } }
    fn update(&mut self, c: u8) {
        self.key0 = crc32(self.key0, c);
        self.key1 = (self.key1.wrapping_add(self.key0 & 0xff))
            .wrapping_mul(0x08088405).wrapping_add(1);
        self.key2 = crc32(self.key2, (self.key1 >> 24) as u8);
    }
    fn decrypt_byte(&self) -> u8 { decrypt_byte(self.key2) }
}

// ===== Attack状態 =====
struct State {
    r: [[u8; 10]; 7],
    c: [[u8; 12]; 7],
    key0_crc:   u32,
    key1_mul1:  u32,
    key1_mul2:  u32,
    key2_crc:   u32,
    s0: [[u8; 2]; 7],
    key0_10lsb: [u8; 7],
    key0_11lsb: [u8; 7],
    key1_10msb: [u8; 7],
    key1_11msb: [u8; 7],
    key0_20lsb: [u8; 7],
    key0_21lsb: [u8; 7],
    key1_20msb: [u8; 7],
    key1_21msb: [u8; 7],
    key0: u32,
    key1: u32,
    key2: u32,
    start_time: Instant,
}

impl State {
    fn new(r: [[u8; 10]; 7], c: [[u8; 12]; 7]) -> Self {
        State {
            r, c,
            key0_crc: 0, key1_mul1: 0, key1_mul2: 0, key2_crc: 0,
            s0: [[0; 2]; 7],
            key0_10lsb: [0; 7], key0_11lsb: [0; 7],
            key1_10msb: [0; 7], key1_11msb: [0; 7],
            key0_20lsb: [0; 7], key0_21lsb: [0; 7],
            key1_20msb: [0; 7], key1_21msb: [0; 7],
            key0: 0, key1: 0, key2: 0,
            start_time: Instant::now(),
        }
    }

    fn attack6(&mut self) -> bool {
        for g0 in 0..0x100u32 {
            for g1 in 0..0x100u32 {
                let key0_h = (g1 << 16) | self.key0_crc;
                let key0 = ((key0_h ^ CRC_TABLE[g0 as usize]) << 8) | g0;
                let mut keys0 = Key::new(key0, self.key1, self.key2);
                let mut keys1 = keys0;
                let mut ok = true;
                for i in 0..10 {
                    let t  = self.c[0][i] ^ keys1.decrypt_byte();
                    keys1.update(t);
                    let t2 = t ^ keys0.decrypt_byte();
                    keys0.update(t2);
                    if t2 != self.r[0][i] { ok = false; break; }
                }
                if ok { self.key0 = key0; return true; }
            }
        }
        false
    }

    fn attack5(&mut self) -> bool {
        for g12 in 0..0x1000000u32 {
            let key1 = (self.key1_mul1 | g12).wrapping_mul(0xd94fa8cd);
            if msb(key1.wrapping_mul(0xd4652819)) == msb(self.key1_mul2) {
                let mut ok = true;
                for f in 0..7 {
                    let k = (key1.wrapping_add(self.key0_10lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_10msb[f] { ok = false; break; }
                    let k = (k.wrapping_add(self.key0_20lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_20msb[f] { ok = false; break; }
                    let k = (key1.wrapping_add(self.key0_11lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_11msb[f] { ok = false; break; }
                    let k = (k.wrapping_add(self.key0_21lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_21msb[f] { ok = false; break; }
                }
                if ok {
                    self.key1 = key1;
                    if self.attack6() { return true; }
                }
            }
        }
        false
    }

    fn attack4(&mut self) -> bool {
        let elapsed = self.start_time.elapsed().as_secs_f64();
        println!(
            "  attack4 {:08x} {:08x} {:08x}  [{:.2}s]",
            self.key0_crc, self.key1_mul2, self.key2_crc, elapsed
        );
        let key2_crc = self.key2_crc;
        let s00      = self.s0[0][0];
        for g11 in 0..0x100u32 {
            let key2 = ((key2_crc ^ CRC_TABLE[g11 as usize]) << 8) | g11;
            if decrypt_byte(key2) == s00 {
                self.key2 = key2;
                if self.attack5() { return true; }
            }
        }
        false
    }

    fn attack3(&mut self, f: usize) -> bool {
        if f == 7 { return self.attack4(); }
        for g0 in 0..2u8 {
            for g1 in 0..2u8 {
                let key0_crc     = self.key0_crc;
                let key2_crc     = self.key2_crc;
                let key1_mul2    = self.key1_mul2;
                let r_f0         = self.r[f][0];
                let r_f1         = self.r[f][1];
                let r_f2         = self.r[f][2];
                let s0_00        = self.s0[0][0];
                let s0_f1        = self.s0[f][1];
                let key0_10lsb_f = self.key0_10lsb[f];
                let key0_11lsb_f = self.key0_11lsb[f];
                let key1_10msb_f = self.key1_10msb[f];
                let key1_11msb_f = self.key1_11msb[f];

                let key0_20lsb = (crc32(key0_crc ^ CRC_TABLE[r_f0 as usize], r_f1) & 0xff) as u8;
                let key1_20msb = ((msb(key1_mul2) as u32)
                    .wrapping_add(0x08)
                    .wrapping_add(g0 as u32)
                    .wrapping_add(msb(
                        (key0_10lsb_f as u32).wrapping_mul(0xd4652819)
                        .wrapping_add((key0_20lsb as u32).wrapping_mul(0x08088405))
                    ) as u32)) as u8;
                let k20 = crc32(key2_crc ^ CRC_TABLE[key1_10msb_f as usize], key1_20msb);
                let s20 = decrypt_byte(k20);

                let key0_21lsb = (crc32(
                    key0_crc ^ CRC_TABLE[(r_f0 ^ s0_00) as usize],
                    r_f1 ^ s0_f1,
                ) & 0xff) as u8;
                let key1_21msb = ((msb(key1_mul2) as u32)
                    .wrapping_add(0x08)
                    .wrapping_add(g1 as u32)
                    .wrapping_add(msb(
                        (key0_11lsb_f as u32).wrapping_mul(0xd4652819)
                        .wrapping_add((key0_21lsb as u32).wrapping_mul(0x08088405))
                    ) as u32)) as u8;
                let k21 = crc32(key2_crc ^ CRC_TABLE[key1_11msb_f as usize], key1_21msb);
                let s21 = decrypt_byte(k21);

                if (r_f2 ^ s20 ^ s21) == self.c[f][2] {
                    self.key0_20lsb[f] = key0_20lsb;
                    self.key1_20msb[f] = key1_20msb;
                    self.key0_21lsb[f] = key0_21lsb;
                    self.key1_21msb[f] = key1_21msb;
                    if self.attack3(f + 1) { return true; }
                }
            }
        }
        false
    }

    fn attack2(&mut self) -> bool {
        let elapsed = self.start_time.elapsed().as_secs_f64();
        println!(
            "attack2 {:08x} {:08x} {:08x}  [{:.2}s]",
            self.key0_crc, self.key1_mul1, self.key2_crc, elapsed
        );
        let base_key0_crc = self.key0_crc;
        let base_key2_crc = self.key2_crc;
        for g0 in 0..0x100u32 {
            self.key0_crc = (base_key0_crc & !0xff00) | (g0 << 8);
            for g1 in 0..0x100u32 {
                self.key1_mul2 = g1 << 24;
                for g2 in 0..0x100u32 {
                    for g3 in 0..4u32 {
                        self.key2_crc = (g2 << 16) | (base_key2_crc & !0xff0003u32) | g3;
                        if self.attack3(0) { return true; }
                    }
                }
            }
        }
        false
    }

    fn attack1(&mut self, f: usize) -> bool {
        if f == 7 { return self.attack2(); }
        for g0 in 0..2u8 {
            for g1 in 0..2u8 {
                let key0_crc  = self.key0_crc;
                let key2_crc  = self.key2_crc;
                let key1_mul1 = self.key1_mul1;
                let r_f0      = self.r[f][0];
                let r_f1      = self.r[f][1];
                let s0_00     = self.s0[0][0];

                let key0_10lsb = (key0_crc ^ CRC_TABLE[r_f0 as usize]) as u8;
                let key1_10msb = ((msb(key1_mul1) as u32)
                    .wrapping_add(msb((key0_10lsb as u32).wrapping_mul(0x08088405)) as u32)
                    .wrapping_add(g0 as u32)) as u8;
                let s0_f1 = decrypt_byte(CRC_TABLE[key1_10msb as usize] ^ key2_crc);

                let key0_11lsb = (key0_crc ^ CRC_TABLE[(r_f0 ^ s0_00) as usize]) as u8;
                let key1_11msb = ((msb(key1_mul1) as u32)
                    .wrapping_add(msb((key0_11lsb as u32).wrapping_mul(0x08088405)) as u32)
                    .wrapping_add(g1 as u32)) as u8;
                let s1_f1 = decrypt_byte(CRC_TABLE[key1_11msb as usize] ^ key2_crc);

                if (r_f1 ^ s0_f1 ^ s1_f1) == self.c[f][1] {
                    self.key0_10lsb[f] = key0_10lsb;
                    self.key1_10msb[f] = key1_10msb;
                    self.s0[f][1]      = s0_f1;
                    self.key0_11lsb[f] = key0_11lsb;
                    self.key1_11msb[f] = key1_11msb;
                    if self.attack1(f + 1) { return true; }
                }
            }
        }
        false
    }

    fn attack0(&mut self) -> bool {
        self.start_time = Instant::now();
        for g0 in 0..0x100u32 {
            self.key0_crc = g0;
            for g1 in 0..0x100u32 {
                self.key1_mul1 = g1 << 24;
                for g2 in 0..0x4000u32 {
                    self.key2_crc = g2 << 2;
                    for g3 in 0..=255u8 {
                        self.s0[0][0] = g3;
                        if self.attack1(0) { return true; }
                    }
                }
            }
        }
        false
    }
}

// ===== ZIPローカルファイルヘッダから暗号化データ開始位置を自動取得 =====
fn read_zip_positions(path: &str) -> Vec<u64> {
    let mut f = File::open(path).expect("zip open failed");
    let mut data = Vec::new();
    f.read_to_end(&mut data).unwrap();

    let mut positions = Vec::new();
    let mut i = 0usize;
    while i + 30 <= data.len() {
        if data[i..i+4] == [0x50, 0x4b, 0x03, 0x04] {
            let fname_len = u16::from_le_bytes([data[i+26], data[i+27]]) as usize;
            let extra_len = u16::from_le_bytes([data[i+28], data[i+29]]) as usize;
            let comp_size = u32::from_le_bytes([data[i+18], data[i+19], data[i+20], data[i+21]]) as usize;
            let enc_start = i + 30 + fname_len + extra_len;
            positions.push(enc_start as u64);
            i = enc_start + comp_size;
        } else {
            i += 1;
        }
    }
    positions
}

fn main() {
    let zip_path = "flag.zip";

    // ── 1. ZIPバイナリから pos を自動取得 ──
    println!("[*] Parsing {} ...", zip_path);
    let positions = read_zip_positions(zip_path);
    assert!(positions.len() >= 7, "Expected >=7 encrypted entries, got {}", positions.len());
    println!("[*] Encrypted header positions:");
    for (i, &p) in positions.iter().enumerate() {
        println!("    [{}] 0x{:x}", i, p);
    }

    // ── 2. 暗号化ヘッダ読み込み ──
    let mut c_data = [[0u8; 12]; 7];
    {
        let mut f = File::open(zip_path).unwrap();
        for i in 0..7 {
            f.seek(SeekFrom::Start(positions[i])).unwrap();
            f.read_exact(&mut c_data[i]).unwrap();
        }
    }
    println!("[*] Cipher headers:");
    for (i, row) in c_data.iter().enumerate() {
        print!("    C[{}]: ", i);
        for b in row.iter() { print!("{:02x} ", b); }
        println!();
    }

    // ── 3. 既知平文 R (guessRand の結果をハードコード) ──
    // C言語の guess_rand.c を実行して得た値をここに貼り付ける
    // C[f][0] との対応確認:
    //   R[0][0]=0x90  C[0][0]=0x90 ✓
    //   R[1][0]=0xcf  C[1][0]=0xcf ✓
    //   R[2][0]=0x70  C[2][0]=0x70 ✓
    //   R[3][0]=0x6b  C[3][0]=0x6b ✓
    //   R[4][0]=0xf2  C[4][0]=0xf2 ✓
    //   R[5][0]=0x31  C[5][0]=0x31 ✓
    //   R[6][0]=0x9e  C[6][0]=0x9e ✓
    let r_data: [[u8; 10]; 7] = [
        [144,  89,  71, 232, 134, 190,  63, 187,  99, 153],  // R[0]
        [207, 209, 189, 126, 170, 203,  34, 140, 192,   1],  // R[1]
        [112,  51,  10, 162, 191,  75, 138,  38, 160,  35],  // R[2]
        [107,  49, 124, 178,  25,   3, 113,  89, 191, 212],  // R[3]
        [242, 142, 166, 176,  13,  80, 123,  48, 221,  60],  // R[4]
        [ 49,  77, 111,  59, 240,  47, 135, 122,  85,  40],  // R[5]
        [158, 192,  89,  27, 115, 115,  31, 228, 205, 222],  // R[6]
    ];

    println!("[*] R values (hardcoded):");
    for (i, row) in r_data.iter().enumerate() {
        print!("    R[{}]: ", i);
        for b in row.iter() { print!("{:02x} ", b); }
        println!();
    }

    // R[f][0] == C[f][0] の整合性確認
    let mut consistent = true;
    for f in 0..7 {
        if r_data[f][0] != c_data[f][0] {
            println!("[!] WARNING: R[{}][0]={:02x} != C[{}][0]={:02x} (R values may be wrong!)",
                f, r_data[f][0], f, c_data[f][0]);
            consistent = false;
        }
    }
    if consistent {
        println!("[*] R[f][0] == C[f][0] for all f: OK");
    }

    // ── 4. Attack ──
    println!("[*] Starting attack ...");
    let mut state = State::new(r_data, c_data);
    let total = Instant::now();

    if state.attack0() {
        println!("\n[+] Keys found!");
        println!("key0: {:08x}", state.key0);
        println!("key1: {:08x}", state.key1);
        println!("key2: {:08x}", state.key2);
    } else {
        println!("[-] Attack failed.");
    }
    println!("[*] Total: {:.2}s", total.elapsed().as_secs_f64());
}

main();

[*] Parsing flag.zip ...
[*] Encrypted header positions:
    [0] 0x43
    [1] 0x141
    [2] 0x23f
    [3] 0x33d
    [4] 0x43b
    [5] 0x539
    [6] 0x637
[*] Cipher headers:
    C[0]: 90 13 5c 03 45 2a 5f 7c 7d 4f 3e dd 
    C[1]: cf 02 70 c9 83 c6 70 e8 0a 33 03 59 
    C[2]: 70 55 42 99 d5 23 08 a1 0a 96 67 21 
    C[3]: 6b 58 d9 c7 c3 0f a9 4d 11 b8 d1 a0 
    C[4]: f2 7d 31 ea 5c 3e 99 f0 e3 f3 31 73 
    C[5]: 31 3a 0d b0 1f a8 c2 5b d1 0c c0 74 
    C[6]: 9e b2 7d 05 f0 fe c7 ea 3f 94 19 e3 
[*] R values (hardcoded):
    R[0]: 90 59 47 e8 86 be 3f bb 63 99 
    R[1]: cf d1 bd 7e aa cb 22 8c c0 01 
    R[2]: 70 33 0a a2 bf 4b 8a 26 a0 23 
    R[3]: 6b 31 7c b2 19 03 71 59 bf d4 
    R[4]: f2 8e a6 b0 0d 50 7b 30 dd 3c 
    R[5]: 31 4d 6f 3b f0 2f 87 7a 55 28 
    R[6]: 9e c0 59 1b 73 73 1f e4 cd de 
[*] R[f][0] == C[f][0] for all f: OK
[*] Starting attack ...
attack2 00000008 00000000 00002ce8  [109.70s]
attack2 0000ff08 00000000 0000ace8  [111.56s]
attack2 0000ff08 80000000 00002

attack2 0000ff60 ff000000 0000d020  [1769.58s]
attack2 00000061 7e000000 00005b30  [1778.33s]
attack2 0000ff61 7e000000 0000db30  [1780.30s]
attack2 0000ff61 fe000000 00005810  [1789.09s]
attack2 0000ff61 fe000000 0000d810  [1791.01s]
attack2 00000062 7e000000 00004364  [1799.74s]
attack2 0000ff62 7e000000 0000c364  [1801.68s]
attack2 0000ff62 7f000000 00004364  [1803.57s]
attack2 0000ff62 7f000000 0000c364  [1805.47s]
attack2 0000ff62 fe000000 00004044  [1814.29s]
attack2 0000ff62 fe000000 0000c044  [1816.17s]
attack2 0000ff62 ff000000 00004044  [1818.06s]
attack2 0000ff62 ff000000 0000c044  [1819.96s]
attack2 00000063 7e000000 00004b54  [1828.68s]
attack2 0000ff63 7e000000 0000cb54  [1830.70s]
attack2 0000ff63 7f000000 00004b54  [1832.57s]
attack2 0000ff63 7f000000 0000cb54  [1834.44s]
attack2 0000ff63 fe000000 00004874  [1843.20s]
attack2 0000ff63 fe000000 0000c874  [1845.08s]
attack2 0000ff63 ff000000 00004874  [1847.02s]
attack2 0000ff63 ff000000 0000c874  [1848.89s]
attack2 00000

attack2 0000ff9f 00000000 0000bfac  [3108.73s]
  attack4 0000889f 04000000 0077bfad  [3109.72s]
  attack4 0000889f 04000000 00f7bfad  [3110.23s]

[+] Keys found!
key0: d0df8cc7
key1: c0187ed5
key2: 2201a0cd
[*] Total: 3110.56s


>Cでは18040.92だったので無茶苦茶速くなった。

```bash
bkcrack -C flag.zip -k d0df8cc7 c0187ed5 2201a0cd -d flag_decrypted.zip
```

In [6]:
use std::fs;

fn main() {
    let dir = "./flag_decrypted";
    
    // 1枚目を読み込んでベースにする
    let mut result = fs::read(format!("{}/flag1.txt", dir)).expect("flag1.txt not found");

    // 2〜7枚目をXOR
    for i in 2..=7 {
        let data = fs::read(format!("{}/flag{}.txt", dir, i))
            .expect(&format!("flag{}.txt not found", i));
        for (r, d) in result.iter_mut().zip(data.iter()) {
            *r ^= d;
        }
    }

    fs::write("flag.txt", result);
}

main()

()

In [2]:
use std::fs::File;
use std::io::{Read, Seek, SeekFrom};
use std::sync::{Arc, Mutex, atomic::{AtomicBool, Ordering}};
use std::time::Instant;

// ===== CRC32テーブル (コンパイル時定数) =====
const CRC_TABLE: [u32; 256] = {
    let mut table = [0u32; 256];
    let mut i = 0;
    while i < 256 {
        let mut c = i as u32;
        let mut j = 0;
        while j < 8 {
            c = if c & 1 != 0 { (c >> 1) ^ 0xedb88320 } else { c >> 1 };
            j += 1;
        }
        table[i] = c;
        i += 1;
    }
    table
};

fn crc32(c: u32, d: u8) -> u32 {
    (c >> 8) ^ CRC_TABLE[((c ^ d as u32) & 0xff) as usize]
}

fn decrypt_byte(key2: u32) -> u8 {
    let temp = (key2 | 2) as u16;
    ((u32::from(temp) * u32::from(temp ^ 1)) >> 8) as u8
}

fn msb(x: u32) -> u8 { (x >> 24) as u8 }

// ===== Key構造体 =====
#[derive(Clone, Copy)]
struct Key { key0: u32, key1: u32, key2: u32 }

impl Key {
    fn new(k0: u32, k1: u32, k2: u32) -> Self { Key { key0: k0, key1: k1, key2: k2 } }
    fn update(&mut self, c: u8) {
        self.key0 = crc32(self.key0, c);
        self.key1 = (self.key1.wrapping_add(self.key0 & 0xff))
            .wrapping_mul(0x08088405).wrapping_add(1);
        self.key2 = crc32(self.key2, (self.key1 >> 24) as u8);
    }
    fn decrypt_byte(&self) -> u8 { decrypt_byte(self.key2) }
}

// ===== スレッド間で共有する入力データ =====
// Clone可能にして各スレッドにコピーして渡す
#[derive(Clone)]
struct Input {
    r: [[u8; 10]; 7],
    c: [[u8; 12]; 7],
}

// ===== 発見した鍵 =====
#[derive(Clone)]
struct FoundKeys {
    key0: u32,
    key1: u32,
    key2: u32,
}

// ===== 1スレッド分の attack 状態 =====
struct State {
    inp: Input,
    tid: usize,   // スレッドID (printの制御用)
    key0_crc:   u32,
    key1_mul1:  u32,
    key1_mul2:  u32,
    key2_crc:   u32,
    s0: [[u8; 2]; 7],
    key0_10lsb: [u8; 7],
    key0_11lsb: [u8; 7],
    key1_10msb: [u8; 7],
    key1_11msb: [u8; 7],
    key0_20lsb: [u8; 7],
    key0_21lsb: [u8; 7],
    key1_20msb: [u8; 7],
    key1_21msb: [u8; 7],
    key0: u32,
    key1: u32,
    key2: u32,
}

impl State {
    fn new(inp: Input, tid: usize) -> Self {
        State {
            inp,
            tid,
            key0_crc: 0, key1_mul1: 0, key1_mul2: 0, key2_crc: 0,
            s0: [[0; 2]; 7],
            key0_10lsb: [0; 7], key0_11lsb: [0; 7],
            key1_10msb: [0; 7], key1_11msb: [0; 7],
            key0_20lsb: [0; 7], key0_21lsb: [0; 7],
            key1_20msb: [0; 7], key1_21msb: [0; 7],
            key0: 0, key1: 0, key2: 0,
        }
    }

    fn attack6(&mut self) -> bool {
        for g0 in 0..0x100u32 {
            for g1 in 0..0x100u32 {
                let key0_h = (g1 << 16) | self.key0_crc;
                let key0 = ((key0_h ^ CRC_TABLE[g0 as usize]) << 8) | g0;
                let mut keys0 = Key::new(key0, self.key1, self.key2);
                let mut keys1 = keys0;
                let mut ok = true;
                for i in 0..10 {
                    let t  = self.inp.c[0][i] ^ keys1.decrypt_byte();
                    keys1.update(t);
                    let t2 = t ^ keys0.decrypt_byte();
                    keys0.update(t2);
                    if t2 != self.inp.r[0][i] { ok = false; break; }
                }
                if ok { self.key0 = key0; return true; }
            }
        }
        false
    }

    fn attack5(&mut self) -> bool {
        for g12 in 0..0x1000000u32 {
            let key1 = (self.key1_mul1 | g12).wrapping_mul(0xd94fa8cd);
            if msb(key1.wrapping_mul(0xd4652819)) == msb(self.key1_mul2) {
                let mut ok = true;
                for f in 0..7 {
                    let k = (key1.wrapping_add(self.key0_10lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_10msb[f] { ok = false; break; }
                    let k = (k.wrapping_add(self.key0_20lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_20msb[f] { ok = false; break; }
                    let k = (key1.wrapping_add(self.key0_11lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_11msb[f] { ok = false; break; }
                    let k = (k.wrapping_add(self.key0_21lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_21msb[f] { ok = false; break; }
                }
                if ok {
                    self.key1 = key1;
                    if self.attack6() { return true; }
                }
            }
        }
        false
    }

    fn attack4(&mut self, start_time: &Instant) -> bool {
        // スレッド0のみprint (他スレッドのprintlnはMutex競合を起こすため)
        if self.tid == 0 {
            let elapsed = start_time.elapsed().as_secs_f64();
            println!(
                "  attack4 {:08x} {:08x} {:08x}  [{:.2}s]",
                self.key0_crc, self.key1_mul2, self.key2_crc, elapsed
            );
        }
        let key2_crc = self.key2_crc;
        let s00      = self.s0[0][0];
        for g11 in 0..0x100u32 {
            let key2 = ((key2_crc ^ CRC_TABLE[g11 as usize]) << 8) | g11;
            if decrypt_byte(key2) == s00 {
                self.key2 = key2;
                if self.attack5() { return true; }
            }
        }
        false
    }

    fn attack3(&mut self, f: usize, start_time: &Instant) -> bool {
        if f == 7 { return self.attack4(start_time); }
        for g0 in 0..2u8 {
            for g1 in 0..2u8 {
                let key0_crc     = self.key0_crc;
                let key2_crc     = self.key2_crc;
                let key1_mul2    = self.key1_mul2;
                let r_f0         = self.inp.r[f][0];
                let r_f1         = self.inp.r[f][1];
                let r_f2         = self.inp.r[f][2];
                let s0_00        = self.s0[0][0];
                let s0_f1        = self.s0[f][1];
                let key0_10lsb_f = self.key0_10lsb[f];
                let key0_11lsb_f = self.key0_11lsb[f];
                let key1_10msb_f = self.key1_10msb[f];
                let key1_11msb_f = self.key1_11msb[f];

                let key0_20lsb = (crc32(key0_crc ^ CRC_TABLE[r_f0 as usize], r_f1) & 0xff) as u8;
                let key1_20msb = ((msb(key1_mul2) as u32)
                    .wrapping_add(0x08)
                    .wrapping_add(g0 as u32)
                    .wrapping_add(msb(
                        (key0_10lsb_f as u32).wrapping_mul(0xd4652819)
                        .wrapping_add((key0_20lsb as u32).wrapping_mul(0x08088405))
                    ) as u32)) as u8;
                let k20 = crc32(key2_crc ^ CRC_TABLE[key1_10msb_f as usize], key1_20msb);
                let s20 = decrypt_byte(k20);

                let key0_21lsb = (crc32(
                    key0_crc ^ CRC_TABLE[(r_f0 ^ s0_00) as usize],
                    r_f1 ^ s0_f1,
                ) & 0xff) as u8;
                let key1_21msb = ((msb(key1_mul2) as u32)
                    .wrapping_add(0x08)
                    .wrapping_add(g1 as u32)
                    .wrapping_add(msb(
                        (key0_11lsb_f as u32).wrapping_mul(0xd4652819)
                        .wrapping_add((key0_21lsb as u32).wrapping_mul(0x08088405))
                    ) as u32)) as u8;
                let k21 = crc32(key2_crc ^ CRC_TABLE[key1_11msb_f as usize], key1_21msb);
                let s21 = decrypt_byte(k21);

                if (r_f2 ^ s20 ^ s21) == self.inp.c[f][2] {
                    self.key0_20lsb[f] = key0_20lsb;
                    self.key1_20msb[f] = key1_20msb;
                    self.key0_21lsb[f] = key0_21lsb;
                    self.key1_21msb[f] = key1_21msb;
                    if self.attack3(f + 1, start_time) { return true; }
                }
            }
        }
        false
    }

    fn attack2(&mut self, start_time: &Instant) -> bool {
        // スレッド0のみprint
        if self.tid == 0 {
            let elapsed = start_time.elapsed().as_secs_f64();
            println!(
                "attack2 {:08x} {:08x} {:08x}  [{:.2}s]",
                self.key0_crc, self.key1_mul1, self.key2_crc, elapsed
            );
        }
        let base_key0_crc = self.key0_crc;
        let base_key2_crc = self.key2_crc;
        for g0 in 0..0x100u32 {
            self.key0_crc = (base_key0_crc & !0xff00) | (g0 << 8);
            for g1 in 0..0x100u32 {
                self.key1_mul2 = g1 << 24;
                for g2 in 0..0x100u32 {
                    for g3 in 0..4u32 {
                        self.key2_crc = (g2 << 16) | (base_key2_crc & !0xff0003u32) | g3;
                        if self.attack3(0, start_time) { return true; }
                    }
                }
            }
        }
        false
    }

    fn attack1(&mut self, f: usize, start_time: &Instant) -> bool {
        if f == 7 { return self.attack2(start_time); }
        for g0 in 0..2u8 {
            for g1 in 0..2u8 {
                let key0_crc  = self.key0_crc;
                let key2_crc  = self.key2_crc;
                let key1_mul1 = self.key1_mul1;
                let r_f0      = self.inp.r[f][0];
                let r_f1      = self.inp.r[f][1];
                let s0_00     = self.s0[0][0];

                let key0_10lsb = (key0_crc ^ CRC_TABLE[r_f0 as usize]) as u8;
                let key1_10msb = ((msb(key1_mul1) as u32)
                    .wrapping_add(msb((key0_10lsb as u32).wrapping_mul(0x08088405)) as u32)
                    .wrapping_add(g0 as u32)) as u8;
                let s0_f1 = decrypt_byte(CRC_TABLE[key1_10msb as usize] ^ key2_crc);

                let key0_11lsb = (key0_crc ^ CRC_TABLE[(r_f0 ^ s0_00) as usize]) as u8;
                let key1_11msb = ((msb(key1_mul1) as u32)
                    .wrapping_add(msb((key0_11lsb as u32).wrapping_mul(0x08088405)) as u32)
                    .wrapping_add(g1 as u32)) as u8;
                let s1_f1 = decrypt_byte(CRC_TABLE[key1_11msb as usize] ^ key2_crc);

                if (r_f1 ^ s0_f1 ^ s1_f1) == self.inp.c[f][1] {
                    self.key0_10lsb[f] = key0_10lsb;
                    self.key1_10msb[f] = key1_10msb;
                    self.s0[f][1]      = s0_f1;
                    self.key0_11lsb[f] = key0_11lsb;
                    self.key1_11msb[f] = key1_11msb;
                    if self.attack1(f + 1, start_time) { return true; }
                }
            }
        }
        false
    }

    // (g0, g1) の組を固定して attack0 の内側を実行
    fn attack0_inner(
        &mut self,
        g0: u32,
        g1: u32,
        start_time: &Instant,
        done: &AtomicBool,
    ) -> bool {
        if done.load(Ordering::Relaxed) { return false; }
        self.key0_crc  = g0;
        self.key1_mul1 = g1 << 24;
        for g2 in 0..0x4000u32 {
            if done.load(Ordering::Relaxed) { return false; }
            self.key2_crc = g2 << 2;
            for g3 in 0..=255u8 {
                self.s0[0][0] = g3;
                if self.attack1(0, start_time) { return true; }
            }
        }
        false
    }
}

// ===== ZIPローカルファイルヘッダから暗号化データ開始位置を自動取得 =====
fn read_zip_positions(path: &str) -> Vec<u64> {
    let mut f = File::open(path).expect("zip open failed");
    let mut data = Vec::new();
    f.read_to_end(&mut data).unwrap();

    let mut positions = Vec::new();
    let mut i = 0usize;
    while i + 30 <= data.len() {
        if data[i..i+4] == [0x50, 0x4b, 0x03, 0x04] {
            let fname_len = u16::from_le_bytes([data[i+26], data[i+27]]) as usize;
            let extra_len = u16::from_le_bytes([data[i+28], data[i+29]]) as usize;
            let comp_size = u32::from_le_bytes([data[i+18], data[i+19], data[i+20], data[i+21]]) as usize;
            let enc_start = i + 30 + fname_len + extra_len;
            positions.push(enc_start as u64);
            i = enc_start + comp_size;
        } else {
            i += 1;
        }
    }
    positions
}

fn main() {
    let zip_path = "flag.zip";

    // ── 1. ZIPバイナリから pos を自動取得 ──
    println!("[*] Parsing {} ...", zip_path);
    let positions = read_zip_positions(zip_path);
    assert!(positions.len() >= 7, "Expected >=7 encrypted entries, got {}", positions.len());
    println!("[*] Encrypted header positions:");
    for (i, &p) in positions.iter().enumerate() {
        println!("    [{}] 0x{:x}", i, p);
    }

    // ── 2. 暗号化ヘッダ読み込み ──
    let mut c_data = [[0u8; 12]; 7];
    {
        let mut f = File::open(zip_path).unwrap();
        for i in 0..7 {
            f.seek(SeekFrom::Start(positions[i])).unwrap();
            f.read_exact(&mut c_data[i]).unwrap();
        }
    }
    println!("[*] Cipher headers:");
    for (i, row) in c_data.iter().enumerate() {
        print!("    C[{}]: ", i);
        for b in row.iter() { print!("{:02x} ", b); }
        println!();
    }

    // ── 3. 既知平文 R (ハードコード) ──
    let r_data: [[u8; 10]; 7] = [
        [144,  89,  71, 232, 134, 190,  63, 187,  99, 153],  // R[0]
        [207, 209, 189, 126, 170, 203,  34, 140, 192,   1],  // R[1]
        [112,  51,  10, 162, 191,  75, 138,  38, 160,  35],  // R[2]
        [107,  49, 124, 178,  25,   3, 113,  89, 191, 212],  // R[3]
        [242, 142, 166, 176,  13,  80, 123,  48, 221,  60],  // R[4]
        [ 49,  77, 111,  59, 240,  47, 135, 122,  85,  40],  // R[5]
        [158, 192,  89,  27, 115, 115,  31, 228, 205, 222],  // R[6]
    ];

    // R[f][0] == C[f][0] 整合性確認
    for f in 0..7 {
        if r_data[f][0] != c_data[f][0] {
            println!("[!] WARNING: R[{}][0]={:02x} != C[{}][0]={:02x}",
                f, r_data[f][0], f, c_data[f][0]);
        }
    }
    println!("[*] R[f][0] == C[f][0] for all f: OK");

    // ── 4. スレッド数の決定 ──
    // 論理CPU数を取得、取得できなければ16を使用
    let num_threads = std::thread::available_parallelism()
        .map(|n| n.get())
        .unwrap_or(16);
    println!("[*] Using {} threads (g0: 0..256 を分割)", num_threads);

    // ── 5. マルチスレッド Attack ──
    // g0*256+g1 の65536個を静的に均等分割してスレッドへ割り当て
    // 環境変数 THREADS で物理コア数に合わせて調整可能
    //   例: THREADS=8 ./stay   (物理コア数に合わせる)
    //       THREADS=16 ./stay  (論理スレッド数)
    let num_threads = std::env::var("THREADS")
        .ok()
        .and_then(|s| s.parse::<usize>().ok())
        .unwrap_or(num_threads);

    let done      = Arc::new(AtomicBool::new(false));
    let result    = Arc::new(Mutex::new(None::<FoundKeys>));
    let start_time = Arc::new(Instant::now());

    // ワークリストを事前に生成: (g0, g1) の全65536ペア
    // スレッドtidは work_id % num_threads == tid の仕事を担当
    let inp = Input { r: r_data, c: c_data };

    println!("[*] Starting attack ({} threads, {} works total) ...", num_threads, 256*256);

    let mut handles = Vec::new();

    for tid in 0..num_threads {
        let inp        = inp.clone();
        let done       = Arc::clone(&done);
        let result     = Arc::clone(&result);
        let start_time = Arc::clone(&start_time);
        let nt         = num_threads as u32;

        let handle = std::thread::spawn(move || {
            let mut state = State::new(inp, tid);

            // work_id = g0*256 + g1 (0..65536) をラウンドロビンで担当
            for work in (tid as u32..65536).step_by(num_threads) {
                if done.load(Ordering::Relaxed) { break; }
                let g0 = work >> 8;
                let g1 = work & 0xff;
                if state.attack0_inner(g0, g1, &start_time, &done) {
                    done.store(true, Ordering::Relaxed);
                    let mut res = result.lock().unwrap();
                    *res = Some(FoundKeys {
                        key0: state.key0,
                        key1: state.key1,
                        key2: state.key2,
                    });
                    break;
                }
            }
            let _ = nt; // suppress unused warning
        });
        handles.push(handle);
    }

    // 全スレッド完了を待機
    for h in handles { h.join().unwrap(); }

    let elapsed = start_time.elapsed().as_secs_f64();

    match result.lock().unwrap().as_ref() {
        Some(keys) => {
            println!("\n[+] Keys found!");
            println!("key0: {:08x}", keys.key0);
            println!("key1: {:08x}", keys.key1);
            println!("key2: {:08x}", keys.key2);
        }
        None => println!("[-] Attack failed."),
    }
    println!("[*] Total: {:.2}s ({} threads)", elapsed, num_threads);
}

main();

[*] Parsing flag.zip ...
[*] Encrypted header positions:
    [0] 0x43
    [1] 0x141
    [2] 0x23f
    [3] 0x33d
    [4] 0x43b
    [5] 0x539
    [6] 0x637
[*] Cipher headers:
    C[0]: 90 13 5c 03 45 2a 5f 7c 7d 4f 3e dd 
    C[1]: cf 02 70 c9 83 c6 70 e8 0a 33 03 59 
    C[2]: 70 55 42 99 d5 23 08 a1 0a 96 67 21 
    C[3]: 6b 58 d9 c7 c3 0f a9 4d 11 b8 d1 a0 
    C[4]: f2 7d 31 ea 5c 3e 99 f0 e3 f3 31 73 
    C[5]: 31 3a 0d b0 1f a8 c2 5b d1 0c c0 74 
    C[6]: 9e b2 7d 05 f0 fe c7 ea 3f 94 19 e3 
[*] R[f][0] == C[f][0] for all f: OK
[*] Using 16 threads (g0: 0..256 を分割)
[*] Starting attack (16 threads, 65536 works total) ...
attack2 00000008 00000000 00002ce8  [18.39s]
attack2 0000ff08 00000000 0000ace8  [23.27s]
attack2 00000008 80000000 00002fc8  [29.27s]
attack2 0000ff08 80000000 0000afc8  [34.29s]
attack2 00000009 00000000 000024d8  [40.45s]
attack2 0000ff09 00000000 0000a4d8  [45.54s]
attack2 00000009 80000000 000027f8  [51.71s]
attack2 0000ff09 80000000 0000a7f8  [56.80s]
attack

attack2 0000ff8f 00000000 0000bc8c  [868.76s]
attack2 0000008f 80000000 00003fac  [871.09s]
attack2 0000ff8f 80000000 0000bfac  [873.10s]
attack2 00000090 00000000 00002a40  [875.41s]
attack2 0000ff90 00000000 000046a0  [877.40s]
attack2 0000ff90 00000000 00ff46a3  [879.39s]
attack2 0000ff90 00000000 0000aa40  [881.33s]
attack2 0000ff90 00000000 0000c6a0  [883.25s]
attack2 0000ff90 00000000 00ffc6a3  [885.19s]
attack2 00000090 80000000 00002960  [887.52s]
attack2 0000ff90 80000000 00004580  [889.51s]
attack2 0000ff90 80000000 00ff4583  [891.40s]
attack2 0000ff90 80000000 0000a960  [893.33s]
attack2 0000ff90 80000000 0000c580  [895.30s]
attack2 0000ff90 80000000 00ffc583  [897.20s]
attack2 00000091 00000000 00002270  [899.52s]
attack2 0000ff91 00000000 00004e90  [901.50s]
attack2 0000ff91 00000000 0000a270  [903.42s]
attack2 0000ff91 00000000 0000ce90  [905.33s]
attack2 00000091 80000000 00002150  [907.69s]
attack2 0000ff91 80000000 00004db0  [909.63s]
attack2 0000ff91 80000000 0000a150

In [4]:
use std::fs::File;
use std::io::{Read, Seek, SeekFrom};
use std::sync::{Arc, Mutex, atomic::{AtomicBool, Ordering}};
use std::time::Instant;

// ===== CRC32テーブル (コンパイル時定数) =====
const CRC_TABLE: [u32; 256] = {
    let mut table = [0u32; 256];
    let mut i = 0;
    while i < 256 {
        let mut c = i as u32;
        let mut j = 0;
        while j < 8 {
            c = if c & 1 != 0 { (c >> 1) ^ 0xedb88320 } else { c >> 1 };
            j += 1;
        }
        table[i] = c;
        i += 1;
    }
    table
};

fn crc32(c: u32, d: u8) -> u32 {
    (c >> 8) ^ CRC_TABLE[((c ^ d as u32) & 0xff) as usize]
}

fn decrypt_byte(key2: u32) -> u8 {
    let temp = (key2 | 2) as u16;
    ((u32::from(temp) * u32::from(temp ^ 1)) >> 8) as u8
}

fn msb(x: u32) -> u8 { (x >> 24) as u8 }

// ===== Key構造体 =====
#[derive(Clone, Copy)]
struct Key { key0: u32, key1: u32, key2: u32 }

impl Key {
    fn new(k0: u32, k1: u32, k2: u32) -> Self { Key { key0: k0, key1: k1, key2: k2 } }
    fn update(&mut self, c: u8) {
        self.key0 = crc32(self.key0, c);
        self.key1 = (self.key1.wrapping_add(self.key0 & 0xff))
            .wrapping_mul(0x08088405).wrapping_add(1);
        self.key2 = crc32(self.key2, (self.key1 >> 24) as u8);
    }
    fn decrypt_byte(&self) -> u8 { decrypt_byte(self.key2) }
}

// ===== スレッド間で共有する入力データ =====
#[derive(Clone)]
struct Input {
    r: [[u8; 10]; 7],
    c: [[u8; 12]; 7],
}

// ===== 発見した鍵 =====
#[derive(Clone)]
struct FoundKeys {
    key0: u32,
    key1: u32,
    key2: u32,
}

// ===== 1スレッド分の attack 状態 =====
struct State {
    inp: Input,
    tid: usize,   // スレッドID (printの制御用: tid==0のみprint)
    key0_crc:   u32,
    key1_mul1:  u32,
    key1_mul2:  u32,
    key2_crc:   u32,
    s0: [[u8; 2]; 7],
    key0_10lsb: [u8; 7],
    key0_11lsb: [u8; 7],
    key1_10msb: [u8; 7],
    key1_11msb: [u8; 7],
    key0_20lsb: [u8; 7],
    key0_21lsb: [u8; 7],
    key1_20msb: [u8; 7],
    key1_21msb: [u8; 7],
    key0: u32,
    key1: u32,
    key2: u32,
}

impl State {
    fn new(inp: Input, tid: usize) -> Self {
        State {
            inp,
            tid,
            key0_crc: 0, key1_mul1: 0, key1_mul2: 0, key2_crc: 0,
            s0: [[0; 2]; 7],
            key0_10lsb: [0; 7], key0_11lsb: [0; 7],
            key1_10msb: [0; 7], key1_11msb: [0; 7],
            key0_20lsb: [0; 7], key0_21lsb: [0; 7],
            key1_20msb: [0; 7], key1_21msb: [0; 7],
            key0: 0, key1: 0, key2: 0,
        }
    }

    fn attack6(&mut self) -> bool {
        for g0 in 0..0x100u32 {
            for g1 in 0..0x100u32 {
                let key0_h = (g1 << 16) | self.key0_crc;
                let key0 = ((key0_h ^ CRC_TABLE[g0 as usize]) << 8) | g0;
                let mut keys0 = Key::new(key0, self.key1, self.key2);
                let mut keys1 = keys0;
                let mut ok = true;
                for i in 0..10 {
                    let t  = self.inp.c[0][i] ^ keys1.decrypt_byte();
                    keys1.update(t);
                    let t2 = t ^ keys0.decrypt_byte();
                    keys0.update(t2);
                    if t2 != self.inp.r[0][i] { ok = false; break; }
                }
                if ok { self.key0 = key0; return true; }
            }
        }
        false
    }

    fn attack5(&mut self) -> bool {
        for g12 in 0..0x1000000u32 {
            let key1 = (self.key1_mul1 | g12).wrapping_mul(0xd94fa8cd);
            if msb(key1.wrapping_mul(0xd4652819)) == msb(self.key1_mul2) {
                let mut ok = true;
                for f in 0..7 {
                    let k = (key1.wrapping_add(self.key0_10lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_10msb[f] { ok = false; break; }
                    let k = (k.wrapping_add(self.key0_20lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_20msb[f] { ok = false; break; }
                    let k = (key1.wrapping_add(self.key0_11lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_11msb[f] { ok = false; break; }
                    let k = (k.wrapping_add(self.key0_21lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k) != self.key1_21msb[f] { ok = false; break; }
                }
                if ok {
                    self.key1 = key1;
                    if self.attack6() { return true; }
                }
            }
        }
        false
    }

    fn attack4(&mut self, start_time: &Instant) -> bool {
        // スレッド0のみprint (複数スレッドからのprintlnはstdoutのMutex競合を起こす)
        if self.tid == 0 {
            let elapsed = start_time.elapsed().as_secs_f64();
            println!(
                "  attack4 {:08x} {:08x} {:08x}  [{:.2}s]",
                self.key0_crc, self.key1_mul2, self.key2_crc, elapsed
            );
        }
        let key2_crc = self.key2_crc;
        let s00      = self.s0[0][0];
        for g11 in 0..0x100u32 {
            let key2 = ((key2_crc ^ CRC_TABLE[g11 as usize]) << 8) | g11;
            if decrypt_byte(key2) == s00 {
                self.key2 = key2;
                if self.attack5() { return true; }
            }
        }
        false
    }

    fn attack3(&mut self, f: usize, start_time: &Instant) -> bool {
        if f == 7 { return self.attack4(start_time); }
        for g0 in 0..2u8 {
            for g1 in 0..2u8 {
                let key0_crc     = self.key0_crc;
                let key2_crc     = self.key2_crc;
                let key1_mul2    = self.key1_mul2;
                let r_f0         = self.inp.r[f][0];
                let r_f1         = self.inp.r[f][1];
                let r_f2         = self.inp.r[f][2];
                let s0_00        = self.s0[0][0];
                let s0_f1        = self.s0[f][1];
                let key0_10lsb_f = self.key0_10lsb[f];
                let key0_11lsb_f = self.key0_11lsb[f];
                let key1_10msb_f = self.key1_10msb[f];
                let key1_11msb_f = self.key1_11msb[f];

                let key0_20lsb = (crc32(key0_crc ^ CRC_TABLE[r_f0 as usize], r_f1) & 0xff) as u8;
                let key1_20msb = ((msb(key1_mul2) as u32)
                    .wrapping_add(0x08)
                    .wrapping_add(g0 as u32)
                    .wrapping_add(msb(
                        (key0_10lsb_f as u32).wrapping_mul(0xd4652819)
                        .wrapping_add((key0_20lsb as u32).wrapping_mul(0x08088405))
                    ) as u32)) as u8;
                let k20 = crc32(key2_crc ^ CRC_TABLE[key1_10msb_f as usize], key1_20msb);
                let s20 = decrypt_byte(k20);

                let key0_21lsb = (crc32(
                    key0_crc ^ CRC_TABLE[(r_f0 ^ s0_00) as usize],
                    r_f1 ^ s0_f1,
                ) & 0xff) as u8;
                let key1_21msb = ((msb(key1_mul2) as u32)
                    .wrapping_add(0x08)
                    .wrapping_add(g1 as u32)
                    .wrapping_add(msb(
                        (key0_11lsb_f as u32).wrapping_mul(0xd4652819)
                        .wrapping_add((key0_21lsb as u32).wrapping_mul(0x08088405))
                    ) as u32)) as u8;
                let k21 = crc32(key2_crc ^ CRC_TABLE[key1_11msb_f as usize], key1_21msb);
                let s21 = decrypt_byte(k21);

                if (r_f2 ^ s20 ^ s21) == self.inp.c[f][2] {
                    self.key0_20lsb[f] = key0_20lsb;
                    self.key1_20msb[f] = key1_20msb;
                    self.key0_21lsb[f] = key0_21lsb;
                    self.key1_21msb[f] = key1_21msb;
                    if self.attack3(f + 1, start_time) { return true; }
                }
            }
        }
        false
    }

    fn attack2(&mut self, start_time: &Instant) -> bool {
        // スレッド0のみprint
        if self.tid == 0 {
            let elapsed = start_time.elapsed().as_secs_f64();
            println!(
                "attack2 {:08x} {:08x} {:08x}  [{:.2}s]",
                self.key0_crc, self.key1_mul1, self.key2_crc, elapsed
            );
        }
        let base_key0_crc = self.key0_crc;
        let base_key2_crc = self.key2_crc;
        for g0 in 0..0x100u32 {
            self.key0_crc = (base_key0_crc & !0xff00) | (g0 << 8);
            for g1 in 0..0x100u32 {
                self.key1_mul2 = g1 << 24;
                for g2 in 0..0x100u32 {
                    for g3 in 0..4u32 {
                        self.key2_crc = (g2 << 16) | (base_key2_crc & !0xff0003u32) | g3;
                        if self.attack3(0, start_time) { return true; }
                    }
                }
            }
        }
        false
    }

    fn attack1(&mut self, f: usize, start_time: &Instant) -> bool {
        if f == 7 { return self.attack2(start_time); }
        for g0 in 0..2u8 {
            for g1 in 0..2u8 {
                let key0_crc  = self.key0_crc;
                let key2_crc  = self.key2_crc;
                let key1_mul1 = self.key1_mul1;
                let r_f0      = self.inp.r[f][0];
                let r_f1      = self.inp.r[f][1];
                let s0_00     = self.s0[0][0];

                let key0_10lsb = (key0_crc ^ CRC_TABLE[r_f0 as usize]) as u8;
                let key1_10msb = ((msb(key1_mul1) as u32)
                    .wrapping_add(msb((key0_10lsb as u32).wrapping_mul(0x08088405)) as u32)
                    .wrapping_add(g0 as u32)) as u8;
                let s0_f1 = decrypt_byte(CRC_TABLE[key1_10msb as usize] ^ key2_crc);

                let key0_11lsb = (key0_crc ^ CRC_TABLE[(r_f0 ^ s0_00) as usize]) as u8;
                let key1_11msb = ((msb(key1_mul1) as u32)
                    .wrapping_add(msb((key0_11lsb as u32).wrapping_mul(0x08088405)) as u32)
                    .wrapping_add(g1 as u32)) as u8;
                let s1_f1 = decrypt_byte(CRC_TABLE[key1_11msb as usize] ^ key2_crc);

                if (r_f1 ^ s0_f1 ^ s1_f1) == self.inp.c[f][1] {
                    self.key0_10lsb[f] = key0_10lsb;
                    self.key1_10msb[f] = key1_10msb;
                    self.s0[f][1]      = s0_f1;
                    self.key0_11lsb[f] = key0_11lsb;
                    self.key1_11msb[f] = key1_11msb;
                    if self.attack1(f + 1, start_time) { return true; }
                }
            }
        }
        false
    }

    // g0 を固定して attack0 の内側を実行 (スレッド分割用)
    // g0 単位 (1/256) で分割するので done チェックは g1 先頭で十分
    fn attack0_inner(
        &mut self,
        g0: u32,
        start_time: &Instant,
        done: &AtomicBool,
    ) -> bool {
        self.key0_crc = g0;
        for g1 in 0..0x100u32 {
            if done.load(Ordering::Relaxed) { return false; }
            self.key1_mul1 = g1 << 24;
            for g2 in 0..0x4000u32 {
                self.key2_crc = g2 << 2;
                for g3 in 0..=255u8 {
                    self.s0[0][0] = g3;
                    if self.attack1(0, start_time) { return true; }
                }
            }
        }
        false
    }
}

// ===== ZIPローカルファイルヘッダから暗号化データ開始位置を自動取得 =====
fn read_zip_positions(path: &str) -> Vec<u64> {
    let mut f = File::open(path).expect("zip open failed");
    let mut data = Vec::new();
    f.read_to_end(&mut data).unwrap();

    let mut positions = Vec::new();
    let mut i = 0usize;
    while i + 30 <= data.len() {
        if data[i..i+4] == [0x50, 0x4b, 0x03, 0x04] {
            let fname_len = u16::from_le_bytes([data[i+26], data[i+27]]) as usize;
            let extra_len = u16::from_le_bytes([data[i+28], data[i+29]]) as usize;
            let comp_size = u32::from_le_bytes([data[i+18], data[i+19], data[i+20], data[i+21]]) as usize;
            let enc_start = i + 30 + fname_len + extra_len;
            positions.push(enc_start as u64);
            i = enc_start + comp_size;
        } else {
            i += 1;
        }
    }
    positions
}

fn main() {
    let zip_path = "flag.zip";

    // ── 1. ZIPバイナリから pos を自動取得 ──
    println!("[*] Parsing {} ...", zip_path);
    let positions = read_zip_positions(zip_path);
    assert!(positions.len() >= 7, "Expected >=7 encrypted entries, got {}", positions.len());
    println!("[*] Encrypted header positions:");
    for (i, &p) in positions.iter().enumerate() {
        println!("    [{}] 0x{:x}", i, p);
    }

    // ── 2. 暗号化ヘッダ読み込み ──
    let mut c_data = [[0u8; 12]; 7];
    {
        let mut f = File::open(zip_path).unwrap();
        for i in 0..7 {
            f.seek(SeekFrom::Start(positions[i])).unwrap();
            f.read_exact(&mut c_data[i]).unwrap();
        }
    }
    println!("[*] Cipher headers:");
    for (i, row) in c_data.iter().enumerate() {
        print!("    C[{}]: ", i);
        for b in row.iter() { print!("{:02x} ", b); }
        println!();
    }

    // ── 3. 既知平文 R (ハードコード) ──
    let r_data: [[u8; 10]; 7] = [
        [144,  89,  71, 232, 134, 190,  63, 187,  99, 153],  // R[0]
        [207, 209, 189, 126, 170, 203,  34, 140, 192,   1],  // R[1]
        [112,  51,  10, 162, 191,  75, 138,  38, 160,  35],  // R[2]
        [107,  49, 124, 178,  25,   3, 113,  89, 191, 212],  // R[3]
        [242, 142, 166, 176,  13,  80, 123,  48, 221,  60],  // R[4]
        [ 49,  77, 111,  59, 240,  47, 135, 122,  85,  40],  // R[5]
        [158, 192,  89,  27, 115, 115,  31, 228, 205, 222],  // R[6]
    ];

    // R[f][0] == C[f][0] 整合性確認
    for f in 0..7 {
        if r_data[f][0] != c_data[f][0] {
            println!("[!] WARNING: R[{}][0]={:02x} != C[{}][0]={:02x}",
                f, r_data[f][0], f, c_data[f][0]);
        }
    }
    println!("[*] R[f][0] == C[f][0] for all f: OK");

    // ── 4. スレッド数の決定 ──
    // 論理CPU数を自動取得、取得できなければ16を使用
    // Jupyter上では let num_threads = 16; のように直接指定も可
    let num_threads = std::thread::available_parallelism()
        .map(|n| n.get())
        .unwrap_or(16);
    println!("[*] Using {} threads (g0: 0..256 をラウンドロビン分割)", num_threads);

    // ── 5. マルチスレッド Attack ──
    // g0 (0..256) を num_threads でラウンドロビン分割
    //   tid=0  → g0=0,  16, 32, ...
    //   tid=1  → g0=1,  17, 33, ...
    //   tid=15 → g0=15, 31, 47, ...
    // 各スレッドは g0 単位で done フラグを確認して早期終了
    // print は tid==0 のスレッドのみ行い stdout の Mutex 競合を排除
    let done       = Arc::new(AtomicBool::new(false));
    let result     = Arc::new(Mutex::new(None::<FoundKeys>));
    let start_time = Arc::new(Instant::now());
    let inp        = Input { r: r_data, c: c_data };

    println!("[*] Starting attack ...");

    let mut handles = Vec::new();

    for tid in 0..num_threads {
        let inp        = inp.clone();
        let done       = Arc::clone(&done);
        let result     = Arc::clone(&result);
        let start_time = Arc::clone(&start_time);

        let handle = std::thread::spawn(move || {
            let mut state = State::new(inp, tid);

            for g0 in (tid as u32..256).step_by(num_threads) {
                if done.load(Ordering::Relaxed) { break; }
                if state.attack0_inner(g0, &start_time, &done) {
                    done.store(true, Ordering::Relaxed);
                    let mut res = result.lock().unwrap();
                    *res = Some(FoundKeys {
                        key0: state.key0,
                        key1: state.key1,
                        key2: state.key2,
                    });
                    break;
                }
            }
        });
        handles.push(handle);
    }

    // 全スレッド完了を待機
    for h in handles { h.join().unwrap(); }

    let elapsed = start_time.elapsed().as_secs_f64();

    match result.lock().unwrap().as_ref() {
        Some(keys) => {
            println!("\n[+] Keys found!");
            println!("key0: {:08x}", keys.key0);
            println!("key1: {:08x}", keys.key1);
            println!("key2: {:08x}", keys.key2);
        }
        None => println!("[-] Attack failed."),
    }
    println!("[*] Total: {:.2}s ({} threads)", elapsed, num_threads);
}

main();

[*] Parsing flag.zip ...
[*] Encrypted header positions:
    [0] 0x43
    [1] 0x141
    [2] 0x23f
    [3] 0x33d
    [4] 0x43b
    [5] 0x539
    [6] 0x637
[*] Cipher headers:
    C[0]: 90 13 5c 03 45 2a 5f 7c 7d 4f 3e dd 
    C[1]: cf 02 70 c9 83 c6 70 e8 0a 33 03 59 
    C[2]: 70 55 42 99 d5 23 08 a1 0a 96 67 21 
    C[3]: 6b 58 d9 c7 c3 0f a9 4d 11 b8 d1 a0 
    C[4]: f2 7d 31 ea 5c 3e 99 f0 e3 f3 31 73 
    C[5]: 31 3a 0d b0 1f a8 c2 5b d1 0c c0 74 
    C[6]: 9e b2 7d 05 f0 fe c7 ea 3f 94 19 e3 
[*] R[f][0] == C[f][0] for all f: OK
[*] Using 16 threads (g0: 0..256 をラウンドロビン分割)
[*] Starting attack ...
attack2 00000010 00000000 000002b8  [36.30s]
attack2 0000ff10 00000000 00006e58  [41.20s]
attack2 0000ff10 00000000 000082b8  [46.19s]
attack2 0000ff10 00000000 0000ee58  [51.28s]
attack2 0000ff10 80000000 00000198  [75.43s]
attack2 0000ff10 80000000 00006d78  [80.48s]
attack2 0000ff10 80000000 00008198  [85.45s]
attack2 0000ff10 80000000 0000ed78  [90.50s]
attack2 00000060 7e000000 00005